## Notes

- Minimum 2 hectares parcels are ok, if near bigger parcels. They might consider using them anyway.
- For Dhar, up to 50m inter-khasra distance is okay.

## Setup

In [ ]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import networkx as nx
import numpy as np
import pandas as pd
import geopandas as gpd


from joblib import Parallel, delayed

# import kml reading and set supported driver
import fiona

fiona.drvsupport.supported_drivers["KML"] = "rw"

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import math
import matplotlib.cm

def generate_colormap(N):
    arr = np.arange(N)/N
    N_up = int(math.ceil(N/7)*7)
    arr.resize(N_up)
    arr = arr.reshape(7,N_up//7).T.reshape(-1)
    ret = matplotlib.cm.hsv(arr)
    n = ret[:,3].size
    a = n//2
    b = n-a
    for i in range(3):
        ret[0:n//2,i] *= np.arange(0.2,1,0.8/a)
    ret[n//2:,3] *= np.arange(1,0.3,-0.7/b)
    return ret

In [ ]:
from gridsample.utils import create_ids, save_shapefiles
# from gridsample.mapping.plot import create_interactive_map

In [ ]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
CLEANED_DATA_DIR = DATA_DIR / "01_processed" / "Solar Parks" / "01 Cleaned Khasras"
OUTPUT_DATA_DIR = DATA_DIR / "01_processed" / "Solar Parks" / "03 Suggested Parcels" / "v2"

## Load cleaned khasras

In [ ]:
# Dhar
dhar_gdf = gpd.read_parquet(CLEANED_DATA_DIR / "dhar_cleaned_khasras.parquet")

In [ ]:
# Sagar
sagar_gdf = gpd.read_parquet(CLEANED_DATA_DIR / "sagar_cleaned_khasras.parquet")
# filter to only the "PA" PAR_TYPE (since it looks like the barren land)
sagar_gdf = sagar_gdf[sagar_gdf["PAR_TYPE"] == "PA"]

# 1. Cluster khasras into parcels

### Functions

In [ ]:
def build_graph_from_gdf_with_distance_threshold(
    gdf,
    distance_threshold=1000,
    n_jobs=-1,
):
    """
    Build a graph from a GeoDataFrame where the nodes are the index of the GeoDataFrame
    and the edges are the distance between the geometries.

    Parameters
    ----------
    gdf : GeoDataFrame
        The GeoDataFrame to build the graph from. Must be in a projected coordinate system.
    distance_threshold : float
        The maximum distance (meters) to build edges between geometries.
    n_jobs : int
        The number of jobs to run in parallel. -1 means using all processors.

    Returns
    -------
    nx.Graph
        The graph with the distances as the edge attribute.
    """

    if gdf.crs == "EPSG:4326":
        raise ValueError("The GeoDataFrame must be in a projected coordinate system.")

    gdf_temp = gdf.copy()
    gdf_temp["row_number"] = np.arange(len(gdf_temp))

    # determine which edges should be added
    def build_edges_to_nearby_geometries(i, geom1, gdf, distance_threshold):
        buffered_geom = geom1.buffer(distance_threshold)
        gdf_intersecting_subset = gdf[gdf.intersects(buffered_geom)]
        edges = []
        for j, geom2 in zip(
            gdf_intersecting_subset["row_number"], gdf_intersecting_subset["geometry"]
        ):
            if i < j:
                distance = geom1.distance(geom2)
                edges.append((gdf_temp.index[i], gdf_temp.index[j], distance))
        return edges

    # apply the function in parallel
    list_of_lists_of_edges = Parallel(n_jobs=n_jobs)(
        delayed(build_edges_to_nearby_geometries)(
            i, geom1, gdf_temp, distance_threshold
        )
        for i, geom1 in zip(gdf_temp["row_number"], gdf_temp["geometry"])
    )

    # flatten the list of lists of edges
    edges = [edge for edges in list_of_lists_of_edges for edge in edges]

    # Build the graph
    G = nx.Graph()
    G.add_nodes_from(gdf_temp.index)
    G.add_weighted_edges_from(edges)

    print(
        f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges."
    )
    return G

In [ ]:
def get_connected_components_by_distance_threshold(G, distance_threshold=None):
    """
    Get the connected components of a graph by a distance threshold. The connected components are
    the nodes that are within the distance threshold of each other.

    Parameters
    ----------
    G : nx.Graph
        The graph to get the connected components from.
    distance_threshold : float, optional
        The distance threshold to use to get the connected components.

    Returns
    -------
    cluster_labels_df : pd.DataFrame
        A DataFrame that maps each node (index) to its cluster label.
    G_filtered_with_cluster_labels : nx.Graph
        The graph with edges filtered by the distance threshold with the cluster labels as node attributes.
    """

    def filter_edges_by_weight(G, max_weight):
        """Helper function - Filter a graph by a maximum weight threshold."""

        # this looping method might be slow. Fix later if need be.
        G_filtered = G.copy()
        for edge in G.edges(data=True):
            if edge[2]["weight"] > max_weight:
                G_filtered.remove_edge(edge[0], edge[1])
        return G_filtered

    # filter to only the edges that are within the distance threshold and get the connected components
    if distance_threshold:
        G_filtered = filter_edges_by_weight(G, max_weight=distance_threshold)
    else:
        G_filtered = G

    # print(len(G.edges), "edges filtered to", len(G_filtered.edges), "by distance threshold", distance_threshold)
    list_of_sets_of_connected_nodes = list(nx.connected_components(G_filtered))

    # create parcel ids
    parcel_ids = create_ids(len(list_of_sets_of_connected_nodes), prefix="PARCEL_")

    # create a dictionary that maps each node to its parcel_id.
    node_to_parcel_id = {}
    for parcel_id, connected_nodes in zip(parcel_ids, list_of_sets_of_connected_nodes):
        for node in connected_nodes:
            node_to_parcel_id[node] = parcel_id

    # create a dataframe that maps each node (as index) to its parcel_id
    cluster_labels_df = pd.DataFrame(
        {"parcel_id": node_to_parcel_id.values()},
        index=list(node_to_parcel_id.keys()),
    )

    # add the parcel_id attribute to the nodes
    G_filtered_with_parcel_id = G_filtered.copy()
    for node in G_filtered_with_parcel_id.nodes:
        G_filtered_with_parcel_id.nodes[node]["parcel_id"] = node_to_parcel_id[node]

    return cluster_labels_df, G_filtered_with_parcel_id

### Apply

In [ ]:
# LOCATION = "Dhar"
# gdf = dhar_gdf.to_crs("24378").reset_index(drop=True)[
#     [
#         "geometry",
#         "khasra_id",
#         "village_name",
#     ]
# ]

In [ ]:
LOCATION = "Sagar"
gdf = sagar_gdf.to_crs("24378").reset_index(drop=True)[
    [
        "geometry",
        "UNQID",
        "village_name",
    ]
]
gdf.rename(columns={"UNQID": "khasra_id"}, inplace=True)

In [ ]:
gdf["Khasra Area (m2)"] = gdf.area

In [ ]:
gdf.plot(column="khasra_id")

In [ ]:
# get graph, only considering neighbours within 1000 meters
G = build_graph_from_gdf_with_distance_threshold(gdf, distance_threshold=200)

In [ ]:
f, ax = plt.subplots(2, 3, figsize=(12, 12))
ax = ax.flatten()

gdf_with_parcel_id = gdf.copy()
G_filtered_dict = {}

# plot original khasras
gdf.plot(ax=ax[0], column="khasra_id", cmap=ListedColormap(generate_colormap(len(gdf))))
ax[0].set_title("Original Khasras (Ungrouped)", fontsize=12)
ax[0].set_xticklabels([])
ax[0].set_yticklabels([])

for i, distance_threshold in enumerate([5, 10, 25, 50, 100]):

    parcel_id_col_name = f"parcel_id_{distance_threshold}m"

    cluster_labels_df, G_filtered_with_parcel_id = (
        get_connected_components_by_distance_threshold(G, distance_threshold)
    )
    G_filtered_dict[distance_threshold] = G_filtered_with_parcel_id
    cluster_labels_df.rename(columns={"parcel_id": parcel_id_col_name}, inplace=True)
    # add parcel_id to gdf
    gdf_with_parcel_id = gdf_with_parcel_id.merge(
        cluster_labels_df, left_index=True, right_index=True, how="left"
    )

    N = len(gdf_with_parcel_id[parcel_id_col_name].unique())

    i = i + 1 # start from second plot position
    gdf_with_parcel_id.plot(column=parcel_id_col_name, ax=ax[i], cmap=ListedColormap(generate_colormap(N)))
    ax[i].set_title(
        f"Max Inter-Khasra Distance: {distance_threshold}m \nNumber of parcels formed: {N}",
        fontsize=12,
    )
    ax[i].set_xticklabels([])
    ax[i].set_yticklabels([])

plt.tight_layout()
plt.savefig(OUTPUT_DATA_DIR / LOCATION / "parcels_w_different_thresholds.png", dpi=300)

#### Do it for the decided threshold

In [ ]:
parcel_id_col_name = "parcel_id"
distance_threshold = 25

cluster_labels_df, G_filtered_with_parcel_id = (
    get_connected_components_by_distance_threshold(G, distance_threshold)
)
G_filtered_dict[distance_threshold] = G_filtered_with_parcel_id
cluster_labels_df.rename(columns={"parcel_id": parcel_id_col_name}, inplace=True)
# add parcel_id to gdf
gdf_with_parcel_id = gdf.merge(
    cluster_labels_df, left_index=True, right_index=True, how="left"
)

In [ ]:
gdf_with_parcel_id.plot(column=parcel_id_col_name, cmap=ListedColormap(generate_colormap(len(gdf_with_parcel_id[parcel_id_col_name].unique()))))

In [ ]:
# get size histogram per parcel id col
gdf_with_parcel_id.groupby("parcel_id")["Khasra Area (m2)"].sum().sort_values(ascending=False)/10_000

In [ ]:
save_shapefiles(
    gdf_with_parcel_id.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION,
    "khasras_with_parcel_id_initial",
    formats=["parquet", "kml", "csv"],
)

### Make parcel-level gdf

In [ ]:
parcel_id_col = "parcel_id"

#### Functions

In [ ]:
def get_closest_parcels(gdf, parcel_id_col="parcel_id"):
    min_distances = []
    closest_ids = []
    for i in range(len(gdf)):
        geom = gdf.iloc[i].geometry
        other_geoms = gdf.drop(gdf.index[i])
        distances = other_geoms.geometry.apply(lambda x: geom.distance(x))
        min_distance = distances.min().round(2)
        closest_id = other_geoms.loc[distances.idxmin()][parcel_id_col]
        min_distances.append(min_distance)
        closest_ids.append(closest_id)
    return min_distances, closest_ids

In [ ]:
def get_intra_parcel_distance_stats(G_filtered, parcel_ids, parcel_id_col="parcel_id"):
    """
    Get the inter-khasra distance stats within each parcel.

    Parameters
    ----------
    G_filtered : nx.Graph
        The graph with the edges filtered by the distance threshold.
    parcel_ids : list
        The list of parcel_ids to get the distances for.

    Returns
    -------
    pd.DataFrame
        A DataFrame with the parcel_id and corresponding distance stats.
    """

    # helper function to get the edge weights of a parcel_id
    def get_edge_weights_by_parcel_id(G_filtered, parcel_id):
        selected_nodes = {
            n for n, d in G_filtered.nodes(data=True) if d.get(parcel_id_col) == parcel_id
        }
        subgraph = G_filtered.subgraph(selected_nodes)
        edge_weights = [d["weight"] for _, _, d in subgraph.edges(data=True)]
        return edge_weights

    distances_list = []
    for parcel_id in parcel_ids:
        distances = get_edge_weights_by_parcel_id(G_filtered, parcel_id)
        if len(distances) == 0:
            avg_distance = 0
            min_distance = 0
            percentile_25th_distance = 0
            percentile_50th_distance = 0
            percentile_75th_distace = 0
            max_distance = 0
        else:
            # avg
            avg_distance = np.mean(distances)
            # min 25% quartile, median, 75% quantile, max
            min_distance = np.min(distances)
            percentile_25th_distance = np.percentile(distances, 20)
            percentile_50th_distance = np.percentile(distances, 50)
            percentile_75th_distace = np.percentile(distances, 75)
            max_distance = np.max(distances)

        distances_list.append({
            parcel_id_col: parcel_id,
            "Inter-Khasra Distance Average (m)": avg_distance,
            "Inter-Khasra Distance Min (m)": min_distance,
            "Inter-Khasra Distance 25th Percentile (m)": percentile_25th_distance,
            "Inter-Khasra Distance Median (m)": percentile_50th_distance,
            "Inter-Khasra Distance 75th Percentile (m)": percentile_75th_distace,
            "Inter-Khasra Distance Max (m)": max_distance,
        })
    
    return pd.DataFrame(distances_list).round(2)

#### Code

In [ ]:
# Group by the cluster value and dissolve to make a new GeoDataFrame. Remove the DISCARDED khasras before this.
parcel_gdf = gdf_with_parcel_id.dissolve(by=parcel_id_col)
parcel_gdf = parcel_gdf.drop(columns=["khasra_id", "Khasra Area (m2)"])
parcel_gdf = parcel_gdf.reset_index()

In [ ]:
parcel_gdf.loc[:, "Original Parcel Area (m2)"] = parcel_gdf["geometry"].area

In [ ]:
# filter to only parcels larger than X
min_parcel_area = 20_000  # 2 hectares
filtered_parcel_gdf = parcel_gdf[parcel_gdf["Original Parcel Area (m2)"] > min_parcel_area]

# relabel them to DISCARDED in the khasra gdf too
too_small_parcel_ids = parcel_gdf[parcel_gdf["Original Parcel Area (m2)"] < min_parcel_area][
    "parcel_id"
].values
filtered_gdf_with_parcel_id = gdf_with_parcel_id.copy()
filtered_gdf_with_parcel_id.loc[
    filtered_gdf_with_parcel_id["parcel_id"].isin(too_small_parcel_ids), "parcel_id"
] = "DISCARDED"

In [ ]:
# add how many khasras are inside
khasra_counts_series = gdf_with_parcel_id.groupby(parcel_id_col)["khasra_id"].count()
filtered_parcel_gdf["Khasra Count"] = filtered_parcel_gdf[parcel_id_col].map(khasra_counts_series)

# add the names of all khasras that fall inside each parcel as a list under khasra_ids
khasra_ids_series = gdf_with_parcel_id.groupby(parcel_id_col)["khasra_id"].apply(list).astype(str)
filtered_parcel_gdf["Khasra IDs"] = filtered_parcel_gdf[parcel_id_col].map(khasra_ids_series)

In [ ]:
# Calculate minimum distances and closest parcel_ids
min_distances, closest_ids = get_closest_parcels(filtered_parcel_gdf, parcel_id_col=parcel_id_col)

# Add the results as new columns
filtered_parcel_gdf.loc[:, "Closest Parcel Distance (m)"] = min_distances
filtered_parcel_gdf.loc[:, "Closest Parcel ID"] = closest_ids

In [ ]:
intra_distances_df = get_intra_parcel_distance_stats(
    G_filtered_with_parcel_id,
    filtered_parcel_gdf[parcel_id_col].unique(),
    parcel_id_col=parcel_id_col,
)

In [ ]:
filtered_parcel_gdf = filtered_parcel_gdf.join(intra_distances_df.set_index(parcel_id_col), on=parcel_id_col)

In [ ]:
filtered_parcel_gdf

In [ ]:
filtered_parcel_gdf.plot(column="parcel_id", legend=True)

In [ ]:
# drop if far and small...

## Save files

In [ ]:
save_shapefiles(
    filtered_parcel_gdf.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION,
    "parcels_filtered",
    formats=["parquet", "kml", "csv"],
)

# 2. Add unusable layers

## Water

In [ ]:
water_bodies_gdf = gpd.read_file(RAW_DATA_DIR / "water" / "DWA Waterbodies Ph1 for Madhya Pradesh.geojson")
water_bodies_gdf = water_bodies_gdf.to_crs("24378")

In [ ]:
water_bodies_gdf

In [ ]:
# get cutout of the water shapes that overlap parcels
water_overlap_gdf = gpd.overlay(
    water_bodies_gdf, filtered_parcel_gdf, how="intersection"
)

In [ ]:
water_overlap_gdf["Unusable Area Water (m2)"] = water_overlap_gdf.area
water_unusable_area_df = water_overlap_gdf.groupby("parcel_id")["Unusable Area Water (m2)"].sum().reset_index()

In [ ]:
ax = filtered_parcel_gdf.plot(figsize=(10, 10), color="brown")
# plot boundaries of different parcel_ids different colors
# boundary_parcel_gdf = filtered_parcel_gdf.copy()
# boundary_parcel_gdf["geometry"] = boundary_parcel_gdf.boundary
# boundary_parcel_gdf.plot(ax=ax, column="parcel_id", cmap=ListedColormap(generate_colormap(len(boundary_parcel_gdf))), linewidth=0.5)
water_overlap_gdf.plot(ax=ax)

### Buildings

In [ ]:
from s2cell.s2cell import lat_lon_to_cell_id
import boto3

#### Download rooftop data

Get the ID of the level 6 S2 Cell that this area sits inside

In [ ]:
s2_ids = []

for index, row in filtered_parcel_gdf.to_crs(4326).iterrows():
    lat = row["geometry"].centroid.y
    lon = row["geometry"].centroid.x
    s2_cell_id = lat_lon_to_cell_id(lat=lat, lon=lon, level=6)
        
    s2_ids.append(s2_cell_id)

s2_ids = list(set(s2_ids))


Download closest S2 cell shapefile from https://beta.source.coop/vida/google-microsoft-open-buildings/geoparquet/by_country_s2/country_iso=IND/

In [ ]:
for s2_cell_id in s2_ids:
    s2_rooftops_path = RAW_DATA_DIR / "rooftops" / f"{s2_cell_id}.parquet"

    if s2_rooftops_path.exists():
        print("File already exists")
    else:
        s3 = boto3.client("s3", endpoint_url="https://data.source.coop")
        s3.download_file(
            "vida",
            f"google-microsoft-open-buildings/geoparquet/by_country_s2/country_iso=IND/{s2_cell_id}.parquet",
            str(s2_rooftops_path),
        )
        print("File downloaded.")

#### Load and process rooftop data

In [ ]:
rooftop_gdf_list = []
for s2_cell_id in s2_ids:

    s2_rooftops_path = RAW_DATA_DIR / "rooftops" / f"{s2_cell_id}.parquet"
    rooftop_gdf = gpd.read_parquet(s2_rooftops_path)
    rooftop_gdf_list.append(rooftop_gdf)

rooftop_gdf = pd.concat(rooftop_gdf_list, ignore_index=True)

In [ ]:
rooftop_gdf = rooftop_gdf[
    [
        "bf_source",
        "confidence",
        "area_in_meters",
        "geometry",
    ]
]

In [ ]:
rooftop_gdf["rooftop_id"] = create_ids(len(rooftop_gdf), f"ROOFTOP_S2_{s2_cell_id}_")

In [ ]:
rooftop_gdf = rooftop_gdf.to_crs(24378)

In [ ]:
subset_rooftops_gdf = rooftop_gdf.sjoin(filtered_parcel_gdf, how="inner", predicate="intersects").drop(columns=["index_right"])
subset_rooftops_gdf.drop(columns=filtered_parcel_gdf.columns.drop("geometry"), inplace=True)

In [ ]:
buffer = 25
buffered_rooftops_gdf = subset_rooftops_gdf.copy()
buffered_rooftops_gdf["geometry"] = buffered_rooftops_gdf.buffer(buffer)

In [ ]:
# get cutout of the water shapes that overlap parcels
buildings_overlap_gdf = gpd.overlay(
    buffered_rooftops_gdf, filtered_parcel_gdf, how="intersection"
)

In [ ]:
buildings_overlap_gdf["Unusable Area Buildings (m2)"] = buildings_overlap_gdf.area
building_unusable_area_df = buildings_overlap_gdf.groupby("parcel_id")["Unusable Area Buildings (m2)"].sum().reset_index()

In [ ]:
ax = filtered_parcel_gdf.plot(figsize=(20,20), color="brown")
water_overlap_gdf.plot(ax=ax)
buildings_overlap_gdf.plot(ax=ax, color="yellow")
# gdf_with_parcel_id.plot(column=parcel_id_col, ax=ax, cmap="jet", alpha=0.5)

## Final Calculations

In [ ]:
usable_filtered_parcel_gdf = filtered_parcel_gdf.copy()

usable_filtered_parcel_gdf = gpd.overlay(
    usable_filtered_parcel_gdf, water_overlap_gdf, how="difference"
)

usable_filtered_parcel_gdf = gpd.overlay(
    usable_filtered_parcel_gdf, buildings_overlap_gdf, how="difference"
)

In [ ]:
usable_filtered_parcel_gdf["Usable Area (m2)"] = usable_filtered_parcel_gdf.area
usable_filtered_parcel_gdf["Unsable Area Total (m2)"] = (
    usable_filtered_parcel_gdf["Original Parcel Area (m2)"]
    - usable_filtered_parcel_gdf["Usable Area (m2)"]
)

# add unusable areas
all_unusable_area_cols_df = water_unusable_area_df.merge(building_unusable_area_df, on="parcel_id", how="outer").fillna(0)
usable_filtered_parcel_gdf.merge(all_unusable_area_cols_df, on="parcel_id", how="left").fillna(0)

In [ ]:
save_shapefiles(
    usable_filtered_parcel_gdf.to_crs(epsg=4326),
    OUTPUT_DATA_DIR / LOCATION,
    "parcels_filtered_usable",
    formats=["parquet", "kml", "csv"],
)

## Scraps

In [ ]:
# def build_graph_from_gdf(gdf):
#     """
#     Build a graph from a GeoDataFrame where the nodes are the index of the GeoDataFrame
#     and the edges are the distance between the geometries.

#     Parameters
#     ----------
#     gdf : GeoDataFrame
#         The GeoDataFrame to build the graph from.

#     Returns
#     -------
#     nx.Graph
#         The graph with the distances as the edge attribute.
#     """

#     gdf_index = list(gdf.index)
#     G = nx.Graph()
#     for i, geom1 in enumerate(gdf.geometry):
#         for j, geom2 in enumerate(gdf.geometry):
#             if i <= j:
#                 distance = geom1.distance(geom2)
#                 G.add_edge(gdf_index[i], gdf_index[j], distance=distance)

#     print(
#         f"Graph built with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges."
#     )
#     return G

In [ ]:
# def calculate_distance_matrix_old(gdf):
#     distances = gdf.geometry.apply(lambda geom1: gdf.geometry.distance(geom1))
#     return distances.to_numpy()

In [ ]:
# def calculate_distance_matrix_upper(gdf):
#     n = len(gdf)
#     distance_matrix_upper = np.full((n, n), np.nan)
#     for i, geom1 in enumerate(gdf.geometry):
#         for j, geom2 in enumerate(gdf.geometry):
#             if i < j:
#                 distance = geom1.distance(geom2)
#                 distance_matrix_upper[i, j] = distance
#                 distance_matrix_upper[j, i] = distance
#             if i == j:
#                 distance_matrix_upper[i, j] = 0
#     return distance_matrix_upper

In [ ]:
# def cluster_adjacent_shapes_old(gdf, distance_threshold):
#     gdf_index = gdf.index

#     # Step 2: Create a distance-filtered spatial graph
#     G = nx.Graph()
#     for i, geom1 in enumerate(gdf.geometry):
#         for j, geom2 in enumerate(gdf.geometry):
#             if i != j:
#                 distance = geom1.distance(geom2)
#                 if distance < distance_threshold:
#                     # this means far away nodes don't actually ever get added to the graph
#                     G.add_edge(i, j, distance=distance)

#     # Step 3: Find connected components
#     connected_components = list(nx.connected_components(G))

#     # Step 4: Convert the connected components to a list of labels that matches the input data
#     data = []
#     for cluster_id, value_set in enumerate(connected_components):
#         for value in value_set:
#             data.append((value, cluster_id))

#     df = pd.DataFrame(data, columns=["index", "cluster"])
#     df.set_index("index", inplace=True)

#     # make index of df the same as gdf and assign any missing values as -1
#     cluster_labels = df.reindex(gdf_index)["cluster"].fillna(-1).astype(int)

#     return cluster_labels, G, connected_components

In [ ]:
# sagar_gdf[sagar_gdf["khasra_id"] == "17677985"].geometry.values[0].distance(
#     sagar_gdf[sagar_gdf["khasra_id"] == "17677984"].geometry.values[0]
# )

In [ ]:
# sagar_gdf_4326 = sagar_gdf.to_crs("4326")
# sagar_gdf_4326["Lat"] = sagar_gdf_4326.centroid.y
# sagar_gdf_4326["Lon"] = sagar_gdf_4326.centroid.x
# create_interactive_map(sagar_gdf_4326, point_id_col="khasra_id", zoom_start=12)

In [ ]:
# from sklearn.cluster import HDBSCAN
# clusters = HDBSCAN(min_cluster_size=2, metric="precomputed", n_jobs=-1).fit(dist_matrix)
# gdf['clustering_dbscan'] = clusters.labels_
# gdf.plot(column='clustering_dbscan', legend=True)